# Davinci Notebook 11 — Evaluation Harness & Baselines

A practical framework for perplexity/loss evaluation, Persian text normalization diagnostics, task scoring, tool-call JSON validation, and reproducible test manifests. This is an evaluation scaffold; benchmark datasets require separate acquisition, licensing, and preprocessing.

In [ ]:
import torch, json, math, re, time, hashlib, unicodedata
from pathlib import Path


In [ ]:
import subprocess, sys\nREPO_ROOT = Path.cwd()\nif not (REPO_ROOT / 'src' / 'davinci').is_dir():\n    REPO_ROOT = Path('/content/Davinci')\n    if not (REPO_ROOT / 'src' / 'davinci').is_dir():\n        subprocess.run(['git', 'clone', 'https://github.com/keyhmoham/Davinci.git', str(REPO_ROOT)], check=True)\nsys.path.insert(0, str(REPO_ROOT))\nprint('Using Davinci source:', REPO_ROOT)\n

## 1. Text normalization and diagnostics

In [ ]:
## 2. Model evaluation: token-weighted loss and perplexity\nThis notebook imports the tested repository implementation rather than maintaining a separate loss loop. Batches may be input-ID tensors, `(input_ids, labels)` pairs, or mappings with `input_ids` and optional `labels`. Labels use `-100` for ignored positions. The helper restores the model's previous train/eval mode.\n

from src.davinci.evaluation import (\n    evaluate_language_model, exact_match_metrics, grouped_exact_match_metrics,\n)\n\ndef evaluate_lm(model, loader, device, max_batches=None):\n    # Canonical implementation handles ignored labels, token-weighting, and mode restoration.\n    return evaluate_language_model(model, loader, device=device, max_batches=max_batches)\n\ndef evaluate_answers(predictions, references, groups):\n    # Group labels must come from the benchmark manifest; do not infer language automatically.\n    return grouped_exact_match_metrics(predictions, references, groups)\n\nprint('Canonical evaluation helpers loaded.')\n

In [ ]:
@torch.no_grad()
def evaluate_lm(model,loader,device,max_batches=None):
 model.eval(); total_loss=0.; total_tokens=0; batches=0
 for batch in loader:
  if max_batches and batches>=max_batches: break
  batch=batch.to(device); logits=model(batch); labels=batch[:,1:]; pred=logits[:,:-1,:].float()
  loss=torch.nn.functional.cross_entropy(pred.reshape(-1,pred.size(-1)),labels.reshape(-1),reduction='sum')
  total_loss+=loss.item(); total_tokens+=labels.numel(); batches+=1
 mean=total_loss/max(1,total_tokens); return {'mean_loss':mean,'perplexity':math.exp(min(80,mean)),'tokens':total_tokens,'batches':batches}


## 3. Tool-call format validation
Use a canonical JSON representation for generated calls; tool execution itself must be sandboxed and permission-controlled.

In [ ]:
def validate_tool_call(obj,available_tools):
 errors=[]
 if not isinstance(obj,dict): return ['Expected object']
 if obj.get('type')!='tool_call': errors.append('type must be tool_call')
 tool=obj.get('name'); args=obj.get('arguments')
 if tool not in available_tools: errors.append('unknown tool name')
 if not isinstance(args,dict): errors.append('arguments must be a JSON object')
 return errors
example={'type':'tool_call','name':'search','arguments':{'query':'example'}}
print('Example validation:',validate_tool_call(example,['search','browser']))


## 4. Benchmark run manifest
Record model revision, tokenizer revision, dataset identifiers, split, prompt format, sampling settings, and evaluator version for reproducibility.

In [ ]:
## 5. Benchmarking checklist\n- Persian, English, and code-mixed results must be reported separately using explicit manifest labels.\n- Include sample counts and uncertainty intervals; do not compare tiny groups as if equally reliable.\n- Long-context retrieval: vary evidence location, distractors, and context length; report each length separately.\n- Tool selection, argument correctness, execution outcomes, and recovery.\n- Safety and refusal quality.\n- Contamination checks and licensing/provenance audit.\n- Report confidence intervals and multiple seeds where practical.\n

## 5. Benchmarking checklist
- Persian language understanding: curated task sets and human review.
- English and bilingual capability: separately scored.
- Long-context retrieval: vary evidence location and distractors.
- Tool selection, arguments, execution outcomes, and recovery.
- Safety and refusal quality.
- Contamination checks and licensing/provenance audit.
- Report confidence intervals and multiple seeds where practical.
